# 方向B · 用英伟达财报训练一个「董秘」

流程（对应课件「深度四」）：
1. 造数据：22→(本作业)62条训练问答，答案注明出处，人工抽检记录见仓库 `data/qa_audit.md`
2. Colab 上用 QLoRA (unsloth) 微调 Qwen3.5-2B（免费 T4 即可）
3. 四格对照：训练过/没训练过 × 原问法/换问法，再和「知识库+同一个小模型」比一比

**运行环境**：Colab → 运行时 → 更改运行时类型 → T4 GPU

**数据进入方式（二选一）**：
- 把仓库推到 GitHub 后改下面的 `REPO_URL` 直接 clone；
- 或运行时用左侧文件面板把 `data/` 目录（4 个 jsonl）拖进 Colab 根目录。

In [ ]:
REPO_URL = "https://github.com/qipeng-ai/homepage-demo.git"

import os, pathlib
!git clone {REPO_URL} repo || true
!if [ -d repo/nvidia-dongmi/data ]; then cp -r repo/nvidia-dongmi/data .; fi
assert pathlib.Path('data/train.jsonl').exists(), '缺少 data/train.jsonl —— 请 clone 仓库或上传 data 目录'
print('数据就绪:', sorted(p.name for p in pathlib.Path('data').glob('*.jsonl')))

In [ ]:
!pip -q install unsloth
# 若此处报错: 菜单 运行时 → 重新启动会话, 再从本格重新运行

In [ ]:
import json, pandas as pd

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding='utf-8')]

train = load_jsonl('data/train.jsonl')
t1    = load_jsonl('data/test_t1.jsonl')            # 训练过 × 换问法
t2    = load_jsonl('data/test_t2.jsonl')            # 没训练过 × 原问法
t2r   = load_jsonl('data/test_t2_rephrased.jsonl')  # 没训练过 × 换问法
print(f'训练 {len(train)} 条 | T1 {len(t1)} | T2 {len(t2)} | T2换问法 {len(t2r)}')
print(train[0]['question'])
print(train[0]['answer'])

## 1. 加载基座模型（先做一轮「微调前」评测，用于对照）

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 1024
try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name='unsloth/Qwen3.5-2B', max_seq_length=MAX_SEQ, load_in_4bit=True)
    BASE_NAME = 'Qwen3.5-2B'
except Exception as e:
    print('Qwen3.5-2B 不可用, 回退 Qwen2.5-1.7B-Instruct:', e)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name='unsloth/Qwen2.5-1.7B-Instruct', max_seq_length=MAX_SEQ, load_in_4bit=True)
    BASE_NAME = 'Qwen2.5-1.7B-Instruct'
print('基座:', BASE_NAME)

In [ ]:
SYS_STYLE = ('你是英伟达公司的董秘。回答规范: 先结论、再细节, 末尾一行【出处】注明财年与章节; '
             '年报里没有的事实不要编造。')

FastLanguageModel.for_inference(model)

def dongmi(model_, tokenizer_, question, max_new_tokens=256):
    prompt = f'{SYS_STYLE}\n\n### 问题：\n{question}\n\n### 回答：\n'
    inputs = tokenizer_(prompt, return_tensors='pt').to(model_.device)
    out = model_.generate(**inputs, max_new_tokens=max_new_tokens,
                          temperature=0.0, do_sample=False)
    text = tokenizer_.decode(out[0], skip_special_tokens=True)
    return text.split('### 回答：')[-1].strip()

# 冒烟测试
print(dongmi(model, tokenizer, '英伟达2026财年总收入是多少？'))

In [ ]:
def eval_grid(model_, tokenizer_):
    """四格对照: 训练过/没训练过 × 原问法/换问法。返回明细 DataFrame。"""
    import random
    random.seed(42)
    groups = [
        ('训练过×原问法',   [dict(question=r['question'], expect=[w for w in _nums(r['answer'])[:2]])
                             for r in random.sample(train, 10)]),
        ('训练过×换问法',   t1),
        ('没训练过×原问法', t2),
        ('没训练过×换问法', t2r),
    ]
    rows = []
    for grid, items in groups:
        for it in items:
            ans = dongmi(model_, tokenizer_, it['question'])
            expect = it.get('expect', [])
            hits = [e for e in expect if e in ans]
            rows.append({'格子': grid, '问题': it['question'],
                         '模型回答': ans,
                         '关键数命中': f'{len(hits)}/{len(expect)}' if expect else '-',
                         '结论对(人工复核)': ''})
    return pd.DataFrame(rows)

import re
def _nums(text):
    return re.findall(r'[\d,.]+(?:%|亿美元|亿|万|美元|倍|个|元)?', text)

before = eval_grid(model, tokenizer)
print('微调前 每格命中情况:')
print(before.groupby('格子')['关键数命中'].apply(lambda s: (s != '0/x').mean().round(2)))

## 2. 挂上 LoRA（冻结主路，只训低秩旁路）

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                               # 课件基线: 秩 r=16
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

def to_text(r):
    return f"{SYS_STYLE}\n\n### 问题：\n{r['question']}\n\n### 回答：\n{r['answer']}"

ds = Dataset.from_list([{ 'text': to_text(r)} for r in train])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ,
    args=SFTConfig(
        output_dir='out',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,            # 62条 x 3轮 ~ 48步
        learning_rate=2e-4,            # 课件基线
        optim='adamw_8bit',
        lr_scheduler_type='cosine',
        logging_steps=5,
        weight_decay=0.01,
        seed=42,
        report_to='none',
    ),
)
trainer.train()   # 免费T4上约几分钟; 盯 loss: 应从 ~3 降到 <1

## 3. 微调后再跑一遍四格对照（与微调前同题）

In [ ]:
FastLanguageModel.for_inference(model)
after = eval_grid(model, tokenizer)

before['阶段'] = '微调前'
after['阶段']  = '微调后'
full = pd.concat([before, after], ignore_index=True)
full.to_csv('四格对照表_明细.csv', index=False)

def hit_rate(df):
    return (df['关键数命中'] != '0/x').mean().round(2)
pivot = full.pivot_table(index='格子', columns='阶段', values='关键数命中', aggfunc=hit_rate)
print('四格对照: 关键数命中率(微调前 -> 微调后)')
print(pivot)

# 抽看几条
for g in ['训练过×换问法', '没训练过×原问法']:
    r = after[after['格子'] == g].iloc[0]
    print('\n' + '=' * 80 + f'\n[{g}] {r["问题"]}\n--- 微调后回答 ---\n{r["模型回答"]}')

## 4. 再和「知识库 + 同一个小模型」比一比（迷你 RAG）

用同一批测试题：微调后裸答 vs 检索增强（把 chunks.jsonl 当知识库，向量召回 top-3 原文块塞进上下文）。

In [ ]:
!pip -q install sentence-transformers
from sentence_transformers import SentenceTransformer
import numpy as np

chunks = load_jsonl('data/chunks.jsonl')
print('知识库块数:', len(chunks))

emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
cvecs = emb.encode([c['text'][:1200] for c in chunks],
                   normalize_embeddings=True, show_progress_bar=True)

def retrieve(question, k=3):
    qv = emb.encode([question], normalize_embeddings=True)
    sims = (cvecs @ qv.T).ravel()
    top = sims.argsort()[::-1][:k]
    return [chunks[i] for i in top]

def dongmi_rag(question, k=3):
    ctx = '\n\n'.join(f"[{c['chunk_id']}] {c['text'][:900]}" for c in retrieve(question, k))
    prompt = (f'{SYS_STYLE}\n\n以下是从年报检索到的原文片段：\n{ctx}\n\n'
              f'### 问题：\n{question}\n\n### 回答：\n')
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=300, temperature=0.0, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True).split('### 回答：')[-1].strip()

# 比一比: T2(没训练过) 这格最有说服力 —— 微调答不出, RAG 能答上
rows = []
for it in t2:
    rows.append({'问题': it['question'],
                 '微调后裸答命中': sum(e in dongmi(model, tokenizer, it['question']) for e in it['expect']),
                 '微调后+RAG命中': sum(e in dongmi_rag(it['question']) for e in it['expect']),
                 '预期关键数': len(it['expect'])})
rag_df = pd.DataFrame(rows)
rag_df.to_csv('RAG对照.csv', index=False)
print(rag_df)
print('\n裸答命中率 %.2f | +RAG 命中率 %.2f' %
      ((rag_df['微调后裸答命中'] / rag_df['预期关键数']).mean(),
       (rag_df['微调后+RAG命中'] / rag_df['预期关键数']).mean()))

## 5. 保存 LoRA 适配器 + 导出交付物

In [ ]:
model.save_pretrained('dongmi_lora_adapter')   # LoRA 适配器(很小)
tokenizer.save_pretrained('dongmi_lora_adapter')

# 打包四格对照表(交作业截图用)
print(pivot)
print('\n明细已保存: 四格对照表_明细.csv / RAG对照.csv / dongmi_lora_adapter/')
print('可从 Colab 左侧文件面板下载 csv; 训练笔记本本文件即可上传 .ipynb 或贴分享链接')